### 1. Installation d'Unsloth

In [1]:
%%capture
import os, re

!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install bitsandbytes

In [2]:
import torch
from google.colab import userdata

hf_token = userdata.get('hf_token')

gpu_props = torch.cuda.get_device_properties(0)
vram_gb = round(gpu_props.total_memory / (1024**3), 2)
bf16_ok = torch.cuda.is_bf16_supported()
print(f"GPU détecté : {gpu_props.name} ({vram_gb} Go VRAM) | Support bfloat16 natif : {bf16_ok}")

GPU détecté : NVIDIA L4 (22.03 Go VRAM) | Support bfloat16 natif : True


### 2. Chargement & Configuration QLoRA

In [3]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import unsloth.models._utils
import unsloth.models.llama

# Neutralise la fonction de télémétrie interne d'Unsloth qui causait le TimeoutError de 120s sur Python 3.13
unsloth.models._utils.get_statistics = lambda *args, **kwargs: None
unsloth.models._utils._download_statistics = lambda *args, **kwargs: None
unsloth.models.llama.get_statistics = lambda *args, **kwargs: None

max_seq_length = 2048
dtype = torch.bfloat16 if is_bfloat16_supported() else None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1112 10:35:04.718000 487 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2026.11.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


Unsloth: compiling each transformer block as one region (MistralDecoderLayer).


### 3. Continued Pre-Training sur le corpus Camfranglais

In [4]:
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer
import json

CPT_HF_REPO = "Cmr237/camfranglais-corpus-cpt"
cpt_dataset = load_dataset(CPT_HF_REPO, split="train", token=hf_token)
print(f"Corpus CPT downloaded from Hugging Face : {len(cpt_dataset)} texts.")

EOS_TOKEN = tokenizer.eos_token

def format_cpt_texts(examples):
    return {"text": [t.strip() + EOS_TOKEN for t in examples["text"]]}

cpt_dataset = cpt_dataset.map(format_cpt_texts, batched=True, num_proc=2)

model.config.use_cache = False
cpt_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = cpt_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.05,
        num_train_epochs = 2,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        save_strategy = "no",
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        dataloader_num_workers = 2,
        seed = 3407,
        output_dir = "outputs_cpt",
        report_to = "none",
    ),
)

cpt_trainer_stats = cpt_trainer.train()

import gc
del cpt_trainer
gc.collect()
torch.cuda.empty_cache()


README.md:   0%|          | 0.00/275 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23224 [00:00<?, ? examples/s]

Corpus CPT downloaded from Hugging Face : 23224 texts.


Map (num_proc=2):   0%|          | 0/23224 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/23224 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/23224 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 296 | Num Epochs = 2 | Total steps = 74
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 114,032,640 of 12,361,815,040 (0.92% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,5.115800
10,4.119100
15,3.849000
20,3.679200
25,3.523700
30,3.640300
35,3.201500
40,3.291800
45,3.358200
50,3.351900


### 4. Supervised Fine-Tuning

On passe maintenant aux dialogues `Cmr237/camfranglais-corpus-sft`

In [5]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="mistral")
SYSTEM_PROMPT = "Tu réponds comme un camerounais"

SFT_HF_REPO = "Cmr237/camfranglais-corpus-sft"
sft_dataset_dict = load_dataset(SFT_HF_REPO, token=hf_token)
sft_train_dataset = sft_dataset_dict["train"]
sft_test_dataset = sft_dataset_dict["test"]
print(f"SFT Dataset Downloaded : {len(sft_train_dataset)} train | {len(sft_test_dataset)} test.")

def formatted_conversations(examples):
    texts = []
    for convo in examples["conversations"]:
        full_convo = [{"role": "system", "content": SYSTEM_PROMPT}] + convo
        texts.append(tokenizer.apply_chat_template(full_convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

sft_train_dataset = sft_train_dataset.map(formatted_conversations, batched=True)
sft_test_dataset = sft_test_dataset.map(formatted_conversations, batched=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_train_dataset,
    eval_dataset = sft_test_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        per_device_eval_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.05,
        num_train_epochs = 2,
        learning_rate = 3e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        save_strategy = "epoch",
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        dataloader_num_workers = 2,
        seed = 3407,
        output_dir = "outputs_sft",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "[INST]",
    response_part = "[/INST]",
)

trainer_stats = trainer.train()

README.md:   0%|          | 0.00/480 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/273k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/32.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2404 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/267 [00:00<?, ? examples/s]

SFT Dataset Downloaded : 2404 train | 267 test.


... (8 outputs left out)

Map:   0%|          | 0/267 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,404 | Num Epochs = 2 | Total steps = 602
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 114,032,640 of 12,361,815,040 (0.92% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.481700
10,3.422000
15,3.596100
20,3.396100
25,3.629100
30,3.463200
35,3.342000
40,3.719200
45,3.585800
50,3.384400


### 5. Évaluation sur les dialogues de `sft_test` & Génération de `test_predictions.json`

In [7]:
import math

eval_metrics = trainer.evaluate()
eval_loss = eval_metrics.get("eval_loss", 0.0)
perplexity = math.exp(eval_loss) if eval_loss < 20 else float("inf")
print(f"Eval Loss on sft_test = {eval_loss:.4f} | Perplexity = {perplexity:.2f}")

FastLanguageModel.for_inference(model)

test_predictions = []
for item in sft_test_dataset:
    convo = item["conversations"]
    user_q = convo[0]["content"]
    human_ref = convo[1]["content"]

    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_q}
    ]
    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        repetition_penalty = 1.08,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    model_ans = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    test_predictions.append({
        "user": user_q,
        "human_reference": human_ref,
        "model_response": model_ans
    })

with open("test_predictions.json", "w", encoding="utf-8") as f:
    json.dump({
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "predictions": test_predictions
    }, f, indent=2, ensure_ascii=False)

print(f"{len(test_predictions)} generated answers saved in test_predictions.json !")

Eval Loss on sft_test = 3.7362 | Perplexity = 41.94
267 generated answers saved in test_predictions.json !


### 6. Inference

In [22]:
FastLanguageModel.for_inference(model)
from transformers import TextStreamer

messages = [{"role": "system", "content": "Tu parles comme un camerounais"}]
while True:
    user_input = str(input("user: \n"))
    if user_input.lower() == "exit":
        break
    messages.append({"role": "user", "content": user_input})
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
        return_dict=True,
    ).to("cuda")

    text_streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True)
    print("Assistant: ", end="", flush=True)
    outputs = model.generate(
        **inputs,
        streamer = text_streamer,
        max_new_tokens = 1024,
        temperature = 0.7,
        repetition_penalty = 1.1,
        do_sample = True,
        pad_token_id = tokenizer.eos_token_id
    )
    # prompt_length = inputs.shape[1]
    response_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    messages.append({"role": "assistant", "content": response_text})

user: 
Yo on dit quoi ? 
Assistant: Ca va bro, et toi?
user: 
posé posé 
Assistant: Justement je voulais te ask nor si tu vois ça comment
user: 
quoi bro dis moi 
Assistant: Je te dis que je ne sais pas encore donc je wanda
user: 
tu ne sais pas encore quoi ?
Assistant: Ce que tu vas me send
user: 
send quoi ? j'ai forget
Assistant: Les photos de la fête là
user: 
quel fête j'ai forget
Assistant: Aïe boss ne me fais pas ça hein
user: 
tu me dis quoi bro
Assistant: Tu es obligé de venir à la fête demain nor
user: 
mais tu m'as dit que la fête est passée norr
Assistant: Non c’est demain
user: 
tu parlais de quelle photo alors ?
Assistant: Celle de la fête de demain
user: 
des photos du futur ??
Assistant: On prend les photos avant l’événement nor !!
user: 
hannnnnn
Assistant: Voilà pourquoi je t’ai ask si tu savais déjà ce que j’allais send
user: 
non garr
Assistant: Tu as tout de suite confirmé alors que tu ne savais pas
user: 
je m'excuse grand
Assistant: Pardon demande à mon gars il t

### 7. Sauvegarde des Adaptateurs LoRA & Export GGUF (`q4_k_m`) vers Hugging Face

In [23]:
model.push_to_hub_gguf(
    "Cmr237/Camfranglais",
    tokenizer,
    quantization_method = ["q4_k_m", "q5_k_m", "q6_k", "q8_0"],
    token = hf_token,
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00005.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/5 [00:00<?, ?it/s]

... (19 outputs left out)

  ...Instruct-2407.Q5_K_M.gguf:   0%|          | 2.68MB / 8.73GB            

  ...o-Instruct-2407.Q6_K.gguf:   0%|          | 11.0MB / 10.1GB            

Unsloth: Successfully uploaded GGUF to https://huggingface.co/Cmr237/Camfranglais
Unsloth: Cleaning up temporary files...


'Cmr237/Camfranglais'